# Load MOTEL schema and produce a simple version

In [1]:
import yaml

## load existing schema

path_schema = r'motel_schema_rev2.yaml'
with open(path_schema, 'r') as f:
    schema = yaml.safe_load(f)
    
schema

{'unmapped_record': {'category': 'record',
  'description': 'Staging schema for raw, unstructured data ingestion. All fields capture free-text inputs before data normalization.',
  'technology_name': {'type': 'string',
   'required': True,
   'description': 'The literal unparsed common name or label of the technology as specified in the raw source material.'},
  'technology': {'type': 'dict',
   'description': 'Contains raw descriptive properties detailing the engineering, structural, and hardware context of the unmapped record.',
   'properties': {'technology_description': {'type': 'string',
     'description': "Free-text engineering parameters describing the technical baseline of the asset (e.g., '5 kW rooftop PV in CH with 18% efficiency')."},
    'technology_type': {'type': 'string',
     'enum': ['conversion', 'storage', 'distribution', 'consumption'],
     'optional': True,
     'description': 'The structural archetype classifying how the physical asset interacts with energy stre

In [14]:
"""
simplify_yaml.py
----------------
Simplifies the MOTEL YAML schema by collapsing 'description', 'type',
'required', 'optional', 'format', 'enum', and 'foreign_key' sibling keys
into a single compact '_description' string on each field node.
"""

from pathlib import Path
import yaml


# ---------------------------------------------------------------------------
# Keys to absorb into '_description'
# ---------------------------------------------------------------------------
INFO_KEYS = ["type", "required", "optional", "format", "enum", "foreign_key", "description"]


def build_info_string(node: dict) -> str:
    """Build a compact info string: 'description text [type, flags, ...]'."""
    brackets = []

    type_val = node.get("type")
    if type_val and type_val != "dict":
        if isinstance(type_val, list):
            brackets.append(f"[{', '.join(str(v) for v in type_val)}]")
        else:
            brackets.append(str(type_val))

    if type_val != "dict":
        if node.get("optional"):
            brackets.append("optional")
        elif "type" in node:
            brackets.append("required")

    fmt = node.get("format")
    if fmt:
        brackets.append(f"format: {fmt}")

    # enum is intentionally omitted

    fk = node.get("foreign_key")
    if fk:
        brackets.append(f"fk -> {fk}")

    desc = node.get("description", "")
    bracket_str = f"[{', '.join(brackets)}]" if brackets else ""
    if desc and bracket_str:
        return f"{bracket_str} {desc}"
    return desc or bracket_str


def is_field_node(node) -> bool:
    """Return True when a dict looks like a field definition (has type/description)."""
    if not isinstance(node, dict):
        return False
    return any(k in node for k in ("type", "description", "required", "optional", "foreign_key", "enum", "format"))


def simplify_node(node):
    """Recursively simplify a parsed YAML node."""
    if isinstance(node, list):
        return [simplify_node(item) for item in node]

    if not isinstance(node, dict):
        return node

    # If this dict looks like a field definition, collapse meta-keys into '_description'
    if is_field_node(node):
        info_str = build_info_string(node)
        # Keep keys that are NOT absorbed into _description
        remaining = {
            k: simplify_node(v)
            for k, v in node.items()
            if k not in INFO_KEYS
        }
        if info_str:
            if not remaining:
                # Solo _description — return the string directly (no wrapping dict)
                return info_str
            result = {"_description": info_str}
            result.update(remaining)
            return result
        return remaining

    # Otherwise recurse into children
    return {k: simplify_node(v) for k, v in node.items()}


# ---------------------------------------------------------------------------
# Custom YAML dumper — keeps '_description' values on one line
# ---------------------------------------------------------------------------
class InlineFriendlyDumper(yaml.Dumper):
    """Dumper that never uses block-scalar style for strings."""
    pass


def _str_representer(dumper, data):
    if "\n" in data:
        return dumper.represent_scalar("tag:yaml.org,2002:str", data, style="|")
    return dumper.represent_scalar("tag:yaml.org,2002:str", data)


InlineFriendlyDumper.add_representer(str, _str_representer)


def dump_yaml(data) -> str:
    return yaml.dump(
        data,
        Dumper=InlineFriendlyDumper,
        default_flow_style=False,
        allow_unicode=True,
        sort_keys=False,
        indent=4,
        width=200,
    )


def flatten_properties(obj):
    if isinstance(obj, dict):
        result = {}

        for key, value in obj.items():
            if key == "properties" and isinstance(value, dict):
                # Promote contents of "properties" to current level
                result.update(flatten_properties(value))
            else:
                result[key] = flatten_properties(value)

        return result

    elif isinstance(obj, list):
        return [flatten_properties(item) for item in obj]

    return obj

# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def simplify_yaml_file(schema, output_path: str, export: bool = True) -> None:
    simplified = simplify_node(schema)
    flattened = flatten_properties(simplified)
    output = dump_yaml(flattened)

    if export:
        Path(output_path).write_text(output, encoding="utf-8")
        print(f"✓ Simplified YAML written to: {output_path}")

    return flattened

OUTPUT_PATH = "motel_simplified.yaml"

schema_rev = simplify_yaml_file(schema, OUTPUT_PATH)
schema_rev

✓ Simplified YAML written to: motel_simplified.yaml


{'unmapped_record': {'_description': 'Staging schema for raw, unstructured data ingestion. All fields capture free-text inputs before data normalization.',
  'category': 'record',
  'technology_name': '[string, required] The literal unparsed common name or label of the technology as specified in the raw source material.',
  'technology': {'_description': 'Contains raw descriptive properties detailing the engineering, structural, and hardware context of the unmapped record.',
   'technology_description': "[string, required] Free-text engineering parameters describing the technical baseline of the asset (e.g., '5 kW rooftop PV in CH with 18% efficiency').",
   'technology_type': '[string, optional] The structural archetype classifying how the physical asset interacts with energy streams.',
   'technology_category': "[string, optional] Broad domain or energy paradigm grouping for the hardware (e.g., 'renewable', 'fossil_fuel', 'nuclear', 'synthetic').",
   'technology_assumption': '[strin

{'unmapped_record': {'_description': 'Staging schema for raw, unstructured data ingestion. All fields capture free-text inputs before data normalization.',
  'category': 'record',
  'technology_name': 'The literal unparsed common name or label of the technology as specified in the raw source material. [string, required]',
  'technology': {'_description': 'Contains raw descriptive properties detailing the engineering, structural, and hardware context of the unmapped record.',
   'technology_description': "Free-text engineering parameters describing the technical baseline of the asset (e.g., '5 kW rooftop PV in CH with 18% efficiency'). [string, required]",
   'technology_type': 'The structural archetype classifying how the physical asset interacts with energy streams. [string, optional]',
   'technology_category': "Broad domain or energy paradigm grouping for the hardware (e.g., 'renewable', 'fossil_fuel', 'nuclear', 'synthetic'). [string, optional]",
   'technology_assumption': 'Explic

In [ ]:
schema_rev = schema.copy()

## rule 1: remove the following keys
keys_to_remove = ['category']

for block_name, block in schema_rev.items():
    for key in keys_to_remove:
        if key in block:
            del block[key]

## rule 2: merge keys (e.g. 'description', 'type', 'required') into a single 'metadata' key
for block_name, block in schema_rev.items():
    metadata = {}
    for key in ['description', 'type', 'required']:
        if key in block:
            metadata[key] = block[key]
            del block[key]
    if metadata:
        block['metadata'] = metadata

schema_rev

{'unmapped_record': {'description': 'Staging schema for raw, unstructured data ingestion. All fields capture free-text inputs before data normalization.',
  'technology_name': {'type': 'string',
   'required': True,
   'description': 'The literal unparsed common name or label of the technology as specified in the raw source material.'},
  'technology': {'type': 'dict',
   'description': 'Contains raw descriptive properties detailing the engineering, structural, and hardware context of the unmapped record.'},
  'scope': {'type': 'dict',
   'description': 'Defines the specific contextual boundaries where the unmapped raw data point remains operationally valid.'},
  'sources': {'type': 'array',
   'description': 'Collection of background reference literature, reports, databases, or creator credentials tracking data origins.',
   'items': {'type': 'dict'}},
  'attributes': {'type': 'array',
   'description': 'Array of technical, financial, or operational metrics populated directly from the